# dynamo_lab：TorchDynamo 图捕获 / Graph Break / Guard 的实测证据

对应文章：[《从 Python 解释器到 PyTorch Dynamo：字节码、Frame 与图捕获》](https://lrypcy.github.io/2026/10/19/python-runtime-dynamo-deepdive/)
（`lrypcy.github.io`，系统实现分类，Python 运行时与图捕获系列第 1 篇）

依赖 `torch 2.10.0`（CPU 即可，不需要 GPU）。固定 `seed = 20261019`，纯 CPU，无网络。
**本文件里嵌的输出来自 CPython 3.11.14 + torch 2.10.0。**

三组（编号接 `runtime_lab.ipynb`）：

7. 从 Python 函数到 FX Graph：捕获了什么
8. Graph Break 触发源清单
9. Guard 的构成 / 失效之后的重编译

每组末尾的 `check()` 就是断言；从上往下 Run All 即可复现全部数字。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
dynamo_lab：TorchDynamo 图捕获 / Graph Break / Guard 的实测证据

对应文章：《从 Python 解释器到 PyTorch Dynamo：字节码、Frame 与图捕获》
（lrypcy.github.io，系统实现分类，Python 运行时与图捕获系列第 1 篇）

依赖：torch 2.10.0（CPU 即可，不需要 GPU）。固定 seed，纯 CPU，无网络。
所有数字都来自本机实测，用来支撑正文里那些「为什么会这样」的判断。

用法：
    python3 dynamo_lab.py            # 跑全部三组
    python3 dynamo_lab.py 7 9        # 只跑第 7、9 组
    python3 dynamo_lab.py --selftest # 只跑断言（静默）
"""

import sys
import warnings

import torch

warnings.filterwarnings("ignore")

from torch._dynamo.testing import EagerAndRecordGraphs  # noqa: E402

OK = []
VERBOSE = True
SEED = 20261019


def check(cond, msg):
    if not cond:
        raise AssertionError(msg)
    OK.append(msg)


def section(title):
    if VERBOSE:
        print("\n" + "=" * 78)
        print(title)
        print("=" * 78)


def record(fn, *args):
    """用 EagerAndRecordGraphs 跑一次，返回 (backend, 输出)。"""
    backend = EagerAndRecordGraphs()
    compiled = torch.compile(fn, backend=backend)
    out = compiled(*args)
    return backend, out


# ----------------------------------------------------------------------------
# G7  从 Python 函数到 FX Graph：捕获了什么
# ----------------------------------------------------------------------------

## G7  从 Python 函数到 FX Graph：捕获了什么

In [2]:
def g7_capture():
    section("G7  从 Python 函数到 FX Graph：捕获了什么")

    torch.manual_seed(SEED)

    def model(x, w):
        y = x @ w
        y = torch.relu(y)
        return y * 2

    x = torch.randn(8, 16)
    w = torch.randn(16, 32)

    backend, out = record(model, x, w)
    n_graphs = len(backend.graphs)

    check(n_graphs == 1, "G7: 纯 Tensor 运算的函数被捕成 1 个图（实得 %d）" % n_graphs)

    gm = backend.graphs[0]
    nodes = [n for n in gm.graph.nodes]
    call_nodes = [n for n in nodes if n.op == "call_function"]
    placeholders = [n for n in nodes if n.op == "placeholder"]
    ops = [n.target.__name__ if hasattr(n.target, "__name__")
           else str(n.target) for n in call_nodes]

    check(ops == ["matmul", "relu", "mul"],
          "G7: 三个算子按数据流排列 matmul → relu → mul（实得 %s）" % ops)
    check(len(placeholders) == 2, "G7: 两个 placeholder 对应 x / w")
    check(torch.allclose(out, torch.relu(x @ w) * 2),
          "G7: 编译后的输出与 eager 一致")

    # 关键：Python 的控制流与赋值没有进图，只留数据依赖
    check(all(n.op != "call_method" for n in nodes),
          "G7: 图里没有 Python 方法调用 —— 只有 Tensor 数据流")

    if VERBOSE:
        print("捕获的图：")
        print(gm.print_readable(print_output=False))
        print("图数量      :", n_graphs)
        print("算子节点    :", ops)
        print("placeholder :", [p.name for p in placeholders])
        print("输出等于 eager:", torch.allclose(out, torch.relu(x @ w) * 2))

    # 带一段纯 Python 逻辑的同一个函数
    def noisy(x):
        y = torch.relu(x)
        print("shape:", tuple(y.shape))
        return y * 2

    backend2, _ = record(noisy, torch.randn(4, 8))
    check(len(backend2.graphs) == 2,
          "G7: 中间插一句 print 之后被切成 2 个图（实得 %d）" % len(backend2.graphs))

    if VERBOSE:
        print("\n带 print 的版本：图数量 =", len(backend2.graphs))
        for i, g in enumerate(backend2.graphs):
            print("  graph%d 算子:" % i,
                  [n.target.__name__ for n in g.graph.nodes if n.op == "call_function"])

g7_capture()


G7  从 Python 函数到 FX Graph：捕获了什么


捕获的图：
class GraphModule(torch.nn.Module):
    def forward(self, L_x_: "f32[8, 16]", L_w_: "f32[16, 32]"):
        l_x_ = L_x_
        l_w_ = L_w_
        
        # File: /var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_41819/1078170853.py:7 in model, code: y = x @ w
        y: "f32[8, 32]" = l_x_ @ l_w_;  l_x_ = l_w_ = None
        
        # File: /var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_41819/1078170853.py:8 in model, code: y = torch.relu(y)
        y_1: "f32[8, 32]" = torch.relu(y);  y = None
        
        # File: /var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T/ipykernel_41819/1078170853.py:9 in model, code: return y * 2
        mul: "f32[8, 32]" = y_1 * 2;  y_1 = None
        return (mul,)
        
图数量      : 1
算子节点    : ['matmul', 'relu', 'mul']
placeholder : ['l_x_', 'l_w_']
输出等于 eager: True
shape: (4, 8)

带 print 的版本：图数量 = 2
  graph0 算子: ['relu']
  graph1 算子: ['mul']


## G8  Graph Break 触发源清单（`torch._dynamo.explain` 实测）

In [3]:
def g8_break_sources():
    section("G8  Graph Break 触发源清单（torch._dynamo.explain 实测）")

    torch.manual_seed(SEED)
    x = torch.randn(8, 8, dtype=torch.float32)

    cases = []

    def clean(t):
        return torch.relu(t) * 2

    def with_print(t):
        y = torch.relu(t)
        print("s", tuple(y.shape))
        return y * 2

    def with_item(t):
        return t.sum().item() * 2

    def data_dep(t):
        y = t @ t
        if y.sum() > 0:
            return torch.relu(y)
        return torch.sigmoid(y)

    def with_tolist(t):
        return [v * 2 for v in t.sum(dim=-1).tolist()]

    def numpy_bridge(t):
        import numpy as np
        return torch.from_numpy(np.asarray(t) * 2)

    for name, fn in [("纯 Tensor 运算（基线）", clean),
                     ("内置 print", with_print),
                     (".item()", with_item),
                     ("数据依赖分支", data_dep),
                     (".tolist()", with_tolist),
                     ("numpy 往返", numpy_bridge)]:
        torch._dynamo.reset()
        exp = torch._dynamo.explain(fn)(x)
        if exp.break_reasons:
            r = exp.break_reasons[0]
            reason = r.reason.strip().splitlines()[0]
            flag = bool(getattr(r, "graph_break", None))
        else:
            reason, flag = "", False
        cases.append((name, exp.graph_count, exp.graph_break_count,
                      exp.op_count, reason, flag))

    # 断言：基线无断点；print 与数据依赖分支各切一刀
    check(cases[0][2] == 0, "G8: 纯 Tensor 基线 0 个断点")
    check(cases[1][1] == 2, "G8: print 把函数切成 2 段（实得 %d）" % cases[1][1])
    check(cases[3][1] == 2, "G8: 数据依赖分支把函数切成 2 段（实得 %d）" % cases[3][1])
    check("generic_jump" in cases[3][4],
          "G8: 数据依赖分支的断点原因是 generic_jump（%s）" % cases[3][4])

    if VERBOSE:
        print("%-22s %6s %6s %6s  %s" % ("用例", "图数", "断点", "算子", "首个断点原因"))
        print("-" * 96)
        for name, gc, bc, oc, reason, flag in cases:
            print("%-22s %6d %6d %6d  %s" % (name, gc, bc, oc, reason[:48]))
        print("\ngraph_break 标志位：", {c[0]: c[5] for c in cases})

    # fullgraph=True：把断点变成硬错误
    torch._dynamo.reset()
    compiled = torch.compile(with_print, fullgraph=True)
    raised = None
    try:
        compiled(torch.randn(4, 4))
    except Exception as exc:  # noqa: BLE001
        raised = type(exc).__name__
    check(raised is not None, "G8: fullgraph=True 时断点变成异常（%s）" % raised)
    if VERBOSE:
        print("\nfullgraph=True 遇到 print 时抛出：", raised)

g8_break_sources()

W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0] Graph break from `Tensor.item()`, consider setting:
W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0] or:
W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0] to include these operations in the captured graph.
W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0] 
W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0] Graph break: from user code at:
W1020 00:12:19.690000 41819 site-packages/torch/_dynamo/variables/tensor.py:1079] [0/0]   File "/var/folders/1h/x16wsv8n5lgf2wz_nlgjjv340000gn/T


G8  Graph Break 触发源清单（torch._dynamo.explain 实测）
s (8, 8)


用例                         图数     断点     算子  首个断点原因
------------------------------------------------------------------------------------------------
纯 Tensor 运算（基线）             1      0      2  
内置 print                    2      1      2  Failed to trace builtin operator
.item()                     1      0      0  Unsupported Tensor.item() call with capture_scal
数据依赖分支                      2      1      3  generic_jump TensorVariable()
.tolist()                   1      0      0  Tensor.tolist() with non-integer tensor
numpy 往返                    1      0      3  

graph_break 标志位： {'纯 Tensor 运算（基线）': False, '内置 print': True, '.item()': True, '数据依赖分支': True, '.tolist()': True, 'numpy 往返': False}

fullgraph=True 遇到 print 时抛出： Unsupported


## G9  Guard 的构成 / 失效之后的重编译

In [4]:
def g9_guards():
    section("G9  Guard 的构成 / 失效之后的重编译")

    torch.manual_seed(SEED)

    def f(x):
        return torch.relu(x) * 2

    # --- guard 清单长什么样 ---
    def dump_guards(fn, *args):
        torch._dynamo.reset()
        exp = torch._dynamo.explain(fn)(*args)
        guards = exp.out_guards
        codes, types = [], set()
        for g in guards:
            for line in (getattr(g, "code_list", None) or []):
                if line:
                    codes.append(line.split(",")[0][:78])
            types.update(getattr(g, "guard_types", None) or [])
        return guards, codes, sorted(types)

    guards, code_lines, types = dump_guards(f, torch.randn(8, 32))

    check(len(guards) >= 6, "G9: 一次最简单的编译也产生 %d 条 guard" % len(guards))
    check("TENSOR_MATCH" in types,
          "G9: Tensor 的形状/dtype 由 TENSOR_MATCH 这类 C++ 侧 guard 检查（%s）" % types)
    check(any("check_obj_id" in c and "torch" in c for c in code_lines),
          "G9: Python 侧 code_list 里有 `torch` 模块的 id 检查")

    def two(x, w):
        y = x @ w
        y = torch.relu(y)
        return y * 2

    guards2, codes2, types2 = dump_guards(
        two, torch.randn(8, 16), torch.randn(16, 32))

    check(len(guards2) > len(guards),
          "G9: 两个 Tensor 输入（%d 条）比一个（%d 条）多出 guard"
          % (len(guards2), len(guards)))
    check(len(codes2) == len(code_lines) + 1,
          "G9: 多出来的正好是第二个输入的 dynamic_indices 检查"
          "（%d 条 code vs %d 条）" % (len(codes2), len(code_lines)))

    if VERBOSE:
        print("单输入 f(x) = relu(x) * 2：")
        print("   guard 总数 %d，带 Python code 的 %d 条" % (len(guards), len(code_lines)))
        print("   guard 类型：", types)
        print("   Python 侧 code_list：")
        for c in code_lines:
            print("      ", c)
        print("\n双输入 model(x, w) = relu(x@w) * 2：")
        print("   guard 总数 %d，带 Python code 的 %d 条" % (len(guards2), len(codes2)))
        print("   guard 类型：", types2)
        print("   Python 侧 code_list：")
        for c in codes2:
            print("      ", c)

    # --- 形状变化 -> 重编译；automatic dynamic shapes 会救一次 ---
    def run_shapes(calls, **cfg):
        torch._dynamo.reset()
        for k, v in cfg.items():
            setattr(torch._dynamo.config, k, v)
        b = EagerAndRecordGraphs()
        c = torch.compile(f, backend=b)
        for t in calls:
            c(t)
        return len(b.graphs)

    four = [torch.randn(n, 32) for n in (4, 8, 16, 32)]
    same = [torch.randn(8, 32) for _ in range(4)]

    n_static = run_shapes(four)
    n_noauto = run_shapes(four, automatic_dynamic_shapes=False)
    n_same = run_shapes(same)

    check(n_same == 1, "G9: 形状不变的 4 次调用只编译 1 次（实得 %d）" % n_same)
    check(n_noauto == 4, "G9: 关掉自动动态后，4 个 batch 各编译一次（实得 %d）" % n_noauto)
    check(1 <= n_static < n_noauto,
          "G9: 默认开启自动动态形状后，4 个 batch 只编译 %d 次 < %d" % (n_static, n_noauto))

    if VERBOSE:
        print("\n4 个不同 batch（4/8/16/32）依次调用：")
        print("   automatic_dynamic_shapes=True （默认）: %d 个图" % n_static)
        print("   automatic_dynamic_shapes=False        : %d 个图" % n_noauto)
        print("4 个相同 batch（8）依次调用                 : %d 个图" % n_same)
        print("\n读法：第一次形状不匹配时 guard 失败触发重编译；")
        print("      自动动态形状会把这一维标记成动态，于是后续尺寸不再重编译。")

    # --- dtype 变化 -> 自动动态救不了 ---
    torch._dynamo.reset()
    b = EagerAndRecordGraphs()
    c = torch.compile(f, backend=b)
    c(torch.randn(8, 32))
    c(torch.randn(8, 32).to(torch.float64))
    c(torch.randn(8, 32))
    n_dtype = len(b.graphs)
    check(n_dtype == 2,
          "G9: dtype 从 f32 换成 f64 再换回来，共 %d 个图（自动动态救不了 dtype）" % n_dtype)
    if VERBOSE:
        print("\ndtype f32 → f64 → f32 三次调用：%d 个图" % n_dtype)

    # --- 动态形状标注 ---
    torch._dynamo.reset()
    b = EagerAndRecordGraphs()
    c = torch.compile(f, backend=b)
    for n in (4, 8, 16, 32):
        t = torch.randn(n, 32)
        torch._dynamo.mark_dynamic(t, 0)
        c(t)
    n_marked = len(b.graphs)
    check(n_marked == 1,
          "G9: 显式 mark_dynamic(t, 0) 后 4 个 batch 只编译 1 次（实得 %d）" % n_marked)
    if VERBOSE:
        print("显式 mark_dynamic(t, 0) 后 4 个 batch：%d 个图" % n_marked)

g9_guards()


G9  Guard 的构成 / 失效之后的重编译
单输入 f(x) = relu(x) * 2：
   guard 总数 10，带 Python code 的 5 条
   guard 类型： ['AUTOGRAD_SAVED_TENSORS_HOOKS', 'DEFAULT_DEVICE', 'ID_MATCH', 'TENSOR_MATCH']
   Python 侧 code_list：
       utils_device.CURRENT_DEVICE == None
       torch._functorch.aot_autograd.utils.top_saved_tensors_hooks ids == None
       ___check_obj_id(G['torch']
       ___check_obj_id(G['torch'].relu
       hasattr(L['x']

双输入 model(x, w) = relu(x@w) * 2：
   guard 总数 11，带 Python code 的 6 条
   guard 类型： ['AUTOGRAD_SAVED_TENSORS_HOOKS', 'DEFAULT_DEVICE', 'ID_MATCH', 'TENSOR_MATCH']
   Python 侧 code_list：
       utils_device.CURRENT_DEVICE == None
       torch._functorch.aot_autograd.utils.top_saved_tensors_hooks ids == None
       hasattr(L['x']
       hasattr(L['w']
       ___check_obj_id(G['torch']
       ___check_obj_id(G['torch'].relu



4 个不同 batch（4/8/16/32）依次调用：
   automatic_dynamic_shapes=True （默认）: 2 个图
   automatic_dynamic_shapes=False        : 4 个图
4 个相同 batch（8）依次调用                 : 1 个图

读法：第一次形状不匹配时 guard 失败触发重编译；
      自动动态形状会把这一维标记成动态，于是后续尺寸不再重编译。



dtype f32 → f64 → f32 三次调用：2 个图
显式 mark_dynamic(t, 0) 后 4 个 batch：1 个图


---

每组末尾的 `check()` 都会把断言记进 `OK`，下面这一格是总数——与 README 和正文里引用的数字对账用。

In [5]:
print("全部通过：%d 条断言" % len(OK))

全部通过：21 条断言
